# Ground Truth Visualization & Dashboard Validator

This notebook replicates the exploratory visualization of `8_26_Brain_Slice_Analysis.ipynb` in a streamlined, lightweight workflow designed to run effortlessly on a laptop:
1. **Direct Data Loading**: Load Maxwell `.raw.h5` recordings via SpikeInterface without pipeline overhead.
2. **Stimulation Bookmarks**: Detect condition blocks (1 Hz, 10 Hz, 20 Hz) to get exact timestamps to navigate on the dashboard.
3. **True Data Reference Graphs**: Plot uncompressed, physical $\mu\text{V}$ traces with electrophysiology L-shaped scale bars.
4. **Dashboard Validation**: Verify that the dashboard viewer renders timestamps, downsampling, and scaling accurately.

In [ ]:
# ==============================================================================
# 1. SETUP & RECORDING LOADING
# ==============================================================================
import os
import sys
from pathlib import Path

# Add src to Python path if running from repo notebooks directory
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from acute_slice_mea.ground_truth import (
    load_recording,
    find_stim_blocks,
    extract_traces,
    plot_ground_truth,
    compare_with_dashboard,
)

# ------------------------------------------------------------------------------
# Specify file paths:
# ------------------------------------------------------------------------------
RAW_PATH = r"C:\Users\dkhun\UCDavis\BenShalomLab\BSlice_raw_data\Experiment4_Slice2_layer1_1minbaseline_1hz_10hz_20hz.raw.h5"
CACHE_DIR = r"cache\Experiment4_Slice2_layer1"

# Load recording (filter_mode: 'raw', 'lfp' [0.5-300 Hz], or 'spike' [300-3000 Hz])
# Set apply_cmr=True if you want global median common referencing applied
rec, fs, meta = load_recording(
    RAW_PATH,
    well_id="well000",
    filter_mode="raw",
    apply_cmr=False,
)

print(f"✓ Loaded Recording: {meta['file_name']}")
print(f"  • Total Channels:   {meta['num_channels']}")
print(f"  • Sampling Rate:    {meta['fs']:.0f} Hz")
print(f"  • Total Duration:   {meta['duration_sec']:.2f} s ({meta['num_frames']} frames)")
print(f"  • Active Filter:    {meta['filter_mode'].upper()} (CMR={meta['cmr_applied']})")

### 2. Detect Stimulation Condition Blocks (Dashboard Bookmark Navigator)
Scans the stimulation channel (e.g. channel `'631'`) to detect pulse trains and group them into condition blocks.
Use these timestamps to navigate to the exact stimulation periods in the dashboard!

In [ ]:
# Detect condition blocks by finding gaps > 4 seconds between pulses
STIM_CHANNEL = "631"
blocks = find_stim_blocks(
    rec,
    stim_channel=STIM_CHANNEL,
    height=500.0,        # Minimum peak height in µV
    gap_threshold_s=4.0, # Gaps longer than 4s demarcate a new condition block
)

print(f"Detected {len(blocks)} Stimulation Condition Blocks on Channel '{STIM_CHANNEL}':")
print(f"{'Block':<8} {'Start Time (s)':>16} {'End Time (s)':>16} {'Duration (s)':>14} {'Pulses':>10}")
print("-" * 68)
for b in blocks:
    print(f"{b['block']:<8} {b['t_start']:>16.2f} {b['t_end']:>16.2f} {b['duration_s']:>14.2f} {b['n_pulses']:>10d}")

### 3. Plot "True Data" Reference Graph
Select the electrodes or channels to inspect, choose a time window in seconds, and render a clean, static figure with L-shaped scale bars.

In [ ]:
# ------------------------------------------------------------------------------
# User Parameters for True Data Visualization
# ------------------------------------------------------------------------------
# Specify channels by string channel_id OR physical electrode_id:
CHANNELS_TO_PLOT = ["947", "566", "8", "631"]   # or: ELECTRODES_TO_PLOT = [821, 148, 532]

# Time window in seconds (automatically converted to sample indices start_frame = int(t0 * fs)):
T_START = 59.8   # e.g., 1-second slice right after 1 Hz block begins
T_END = 60.8

# Visual parameters:
VOLTAGE_LIMITS = (-150, 150)  # (y_min, y_max) in µV (or None for auto)
SPACING_UV = 250.0            # Vertical trace separation in µV (matches dashboard)
DISPLAY_GAIN = 1.0            # Display gain multiplier

# Scale bar sizes:
SHOW_SCALEBARS = True
SCALEBAR_TIME_MS = 10.0       # Horizontal scale bar (ms)
SCALEBAR_VOLTAGE_UV = 50.0    # Vertical scale bar (µV)

# ------------------------------------------------------------------------------
# Extract and Plot:
# ------------------------------------------------------------------------------
payloads = extract_traces(
    rec,
    channels=CHANNELS_TO_PLOT,
    t0=T_START,
    t1=T_END,
    cache_dir=CACHE_DIR if Path(CACHE_DIR).exists() else None,
    baseline_subtract=True,
)

fig = plot_ground_truth(
    payloads,
    spacing_uv=SPACING_UV,
    gain=DISPLAY_GAIN,
    xlim=(T_START, T_END),
    ylim=VOLTAGE_LIMITS,
    scalebars=SHOW_SCALEBARS,
    scalebar_ms=SCALEBAR_TIME_MS,
    scalebar_uv=SCALEBAR_VOLTAGE_UV,
    title=f"Ground Truth Reference Traces [{T_START:.2f}s – {T_END:.2f}s]",
    save_path="reports/ground_truth_sample.png",
    figsize=(11, 6),
)

### 4. Optional: Direct Comparison Against Dashboard Data
Validates that the dashboard (`src/acute_slice_mea/dashboard.py` / `data_loader.py`) renders this exact segment without downsampling distortions or scaling discrepancies.

In [ ]:
# Compare a specific electrode against dashboard output
TARGET_ELECTRODE = 821  # Physical electrode pad ID

try:
    fig_comp, metrics = compare_with_dashboard(
        RAW_PATH,
        CACHE_DIR,
        electrodes=[TARGET_ELECTRODE],
        t0=T_START,
        t1=T_END,
        signal="raw",  # 'raw' or 'lfp'
        gain=DISPLAY_GAIN,
        save_path="reports/dashboard_validation_check.png",
    )
    print("Validation Metrics:")
    print(f"  • Max Absolute Error: {metrics['max_absolute_error_uv']:.4f} µV")
    print(f"  • RMS Error:          {metrics['rms_error_uv']:.4f} µV")
    print(f"  • Pearson r:          {metrics['pearson_correlation']:.5f}")
except Exception as exc:
    print(f"Comparison skipped or cache not ready: {exc}")